# Value Iteration Grid World

This notebook is a **student-friendly companion** to the Value Iteration video.

The goal is to let you run the algorithm **one sweep at a time** and clearly see:

- how the value table changes after each Bellman update
- how information spreads backward from the goal
- when the value function has converged
- what greedy policy is implied by the current values
- how an agent moves under the current or final value function

This **fixed version** makes the step-by-step updates much clearer than before:

- every sweep prints the **before / after / delta** tables
- the plot title shows the **current sweep number**
- a helper called `step_and_render()` does **update + display** in one command
- the notebook warns you when you are looking at the **current** value function instead of the **final** one


## How this notebook matches the video

This notebook keeps the same learning setup as the animation:

- 5 × 5 grid world
- discount factor `gamma = 0.9`
- convergence threshold `epsilon = 0.1`
- the same reward layout:
  - Goal = `+1.0`
  - Trap = `-1.0`
  - Mud = `-0.5`
  - Normal step = `-0.04`

One important teaching fix is included here:

In the original animation code, the Goal is *described* like a terminal state, but the sweep logic can still update it like a normal state.  
In this notebook, the Goal is treated as a **true terminal state** with fixed value `0.0`.

That way, when you move **into** the goal you still get reward `+1.0`, but the value function does not blow up.


In [ ]:
# Run this cell first
%matplotlib inline


## 1. Main explorer class

This class is written for students, so it includes a few helpers that make the algorithm easier to follow:

- `step_and_render()` → do one Bellman sweep and immediately show the result
- `show_status()` → print the current sweep number and convergence state
- `values_table()` → see the values in a clean table
- `policy_table()` → see the greedy policy implied by the current values
- `inspect_state((r, c))` → inspect one state's one-step lookahead table


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ARROWS = {0: "↑", 1: "↓", 2: "←", 3: "→"}
ACTION_NAMES = {0: "Up", 1: "Down", 2: "Left", 3: "Right"}
ACTION_DELTAS = {
    0: (-1, 0),  # Up
    1: (1, 0),   # Down
    2: (0, -1),  # Left
    3: (0, 1),   # Right
}

class ValueIterationExplorer:
    """
    Student-facing Value Iteration explorer for the 5x5 grid world.
    """

    def __init__(self, gamma=0.9, epsilon=0.1, seed=42, max_iterations=40):
        self.gamma = gamma
        self.epsilon = epsilon
        self.seed = seed
        self.max_iterations = max_iterations

        self.grid_size = 5
        self.start_state = (0, 0)
        self.goal_state = (4, 4)
        self.demo_target_state = (3, 4)

        self.rewards = np.array([
            [-0.04, -0.04, -0.04, -0.50, -1.00],
            [-0.04, -1.00, -0.04, -1.00, -1.00],
            [-0.04, -0.50, -0.04, -0.04, -0.04],
            [-1.00, -1.00, -1.00, -1.00, -0.04],
            [-1.00, -0.04, -0.04, -0.04,  1.00],
        ], dtype=float)

        self._build_color_grid()
        self.reset(seed=seed)

    def _build_color_grid(self):
        self.color_grid = np.zeros((self.grid_size, self.grid_size, 3), dtype=float)
        normal = np.array([65, 77, 99]) / 255.0
        mud = np.array([242, 106, 10]) / 255.0
        trap = np.array([145, 25, 25]) / 255.0
        goal = np.array([24, 128, 120]) / 255.0

        for r in range(self.grid_size):
            for c in range(self.grid_size):
                reward = float(self.rewards[r, c])
                if reward == 1.0:
                    self.color_grid[r, c] = goal
                elif reward == -1.0:
                    self.color_grid[r, c] = trap
                elif reward == -0.5:
                    self.color_grid[r, c] = mud
                else:
                    self.color_grid[r, c] = normal

    def reset(self, seed=None):
        if seed is not None:
            self.seed = seed

        self.values = np.zeros((self.grid_size, self.grid_size), dtype=float)
        self.previous_values = None
        self.last_delta_grid = np.zeros_like(self.values)
        self.iteration = 0
        self.error_history = []
        self.last_max_error = None
        self.converged = False
        self.current_phase = "Initialization"
        print("Environment reset. Values start from all zeros.")

    def _reward_label(self, reward):
        if reward == 1.0:
            return "Goal\n+1.0"
        if reward == -1.0:
            return "Trap\n-1.0"
        if reward == -0.5:
            return "Mud\n-0.5"
        return "-0.04"

    def _transition(self, r, c, action_idx, values_snapshot=None):
        if values_snapshot is None:
            values_snapshot = self.values

        if (r, c) == self.goal_state:
            return {
                "next_state": self.goal_state,
                "reward": 0.0,
                "next_value": 0.0,
                "q": 0.0,
                "hit_wall": False,
            }

        dr, dc = ACTION_DELTAS[action_idx]
        nr, nc = r + dr, c + dc
        hit_wall = False

        if nr < 0 or nr >= self.grid_size or nc < 0 or nc >= self.grid_size:
            nr, nc = r, c
            hit_wall = True

        reward = float(self.rewards[nr, nc])

        if (nr, nc) == self.goal_state:
            next_value = 0.0
        else:
            next_value = float(values_snapshot[nr, nc])

        q = reward + self.gamma * next_value

        return {
            "next_state": (nr, nc),
            "reward": reward,
            "next_value": next_value,
            "q": q,
            "hit_wall": hit_wall,
        }

    def one_step_lookahead(self, state=None, values_snapshot=None):
        if state is None:
            state = self.demo_target_state
        if values_snapshot is None:
            values_snapshot = self.values

        r, c = state

        if state == self.goal_state:
            return pd.DataFrame([{
                "action": "Terminal",
                "next_state": self.goal_state,
                "reward": 0.0,
                "next_value": 0.0,
                "q_value": 0.0,
                "wall?": False,
                "note": "Goal is terminal",
            }])

        rows = []
        for idx in range(4):
            tr = self._transition(r, c, idx, values_snapshot)
            rows.append({
                "action": ACTION_NAMES[idx],
                "next_state": tr["next_state"],
                "reward": tr["reward"],
                "next_value": tr["next_value"],
                "q_value": tr["q"],
                "wall?": tr["hit_wall"],
            })

        return pd.DataFrame(rows).sort_values("q_value", ascending=False).reset_index(drop=True)

    def greedy_actions(self, state):
        q_df = self.one_step_lookahead(state)
        if "q_value" not in q_df.columns:
            return [], []

        q_values = np.array([self._transition(state[0], state[1], idx)["q"] for idx in range(4)], dtype=float)
        best_q = float(np.max(q_values))
        best_indices = [idx for idx, q in enumerate(q_values) if np.isclose(q, best_q)]
        return best_indices, q_values

    def values_table(self, values=None, decimals=3):
        if values is None:
            values = self.values
        df = pd.DataFrame(
            np.round(values, decimals),
            index=[f"r{i}" for i in range(self.grid_size)],
            columns=[f"c{j}" for j in range(self.grid_size)],
        )
        return df

    def policy_table(self):
        rows = []
        for r in range(self.grid_size):
            row = []
            for c in range(self.grid_size):
                if (r, c) == self.goal_state:
                    row.append("Goal")
                else:
                    best_indices, _ = self.greedy_actions((r, c))
                    row.append(" / ".join(ARROWS[idx] for idx in best_indices))
            rows.append(row)

        return pd.DataFrame(
            rows,
            index=[f"r{i}" for i in range(self.grid_size)],
            columns=[f"c{j}" for j in range(self.grid_size)],
        )

    def show_status(self):
        print(f"Current phase: {self.current_phase}")
        print(f"Sweep number: {self.iteration}")
        print(f"Gamma: {self.gamma}")
        print(f"Epsilon: {self.epsilon}")
        if self.last_max_error is None:
            print("Last max error: not available yet")
        else:
            print(f"Last max error: {self.last_max_error:.4f}")
        print(f"Converged: {self.converged}")

    def inspect_state(self, state=None):
        if state is None:
            state = self.demo_target_state
        print(f"One-step lookahead for state {state}:")
        df = self.one_step_lookahead(state)
        display(df)
        return df

    def iterate_once(self, verbose=True):
        if self.converged:
            if verbose:
                print("The value function is already below the threshold.")
                print("You can still render it, inspect states, or plot the agent run.")
            return self.last_max_error

        snapshot = self.values.copy()
        new_values = snapshot.copy()
        self.current_phase = "Bellman Update"

        for r in range(self.grid_size):
            for c in range(self.grid_size):
                if (r, c) == self.goal_state:
                    new_values[r, c] = 0.0
                    continue

                q_df = self.one_step_lookahead((r, c), snapshot)
                new_values[r, c] = float(q_df["q_value"].max())

        delta = new_values - snapshot
        max_error = float(np.max(np.abs(delta)))

        self.previous_values = snapshot
        self.last_delta_grid = delta
        self.values = new_values
        self.iteration += 1
        self.last_max_error = max_error
        self.error_history.append(max_error)
        self.converged = max_error < self.epsilon

        if verbose:
            changed_cells = int(np.sum(np.abs(delta) > 1e-12))
            print(f"Completed sweep {self.iteration}. Max error = {max_error:.4f}.")
            print(f"Changed cells this sweep: {changed_cells}")
            if self.converged:
                print(f"Converged because max error < epsilon ({self.epsilon}).")
            else:
                print(f"Not converged yet. We still need max error < {self.epsilon}.")

        return max_error

    def step_and_render(self, show_tables=True, show_policy=False, decimals=3, figsize=(14, 6)):
        self.iterate_once(verbose=True)

        if show_tables:
            print("\nValues BEFORE this sweep:")
            if self.previous_values is None:
                display(self.values_table())
            else:
                display(self.values_table(self.previous_values, decimals=decimals))

            print("\nValues AFTER this sweep:")
            display(self.values_table(self.values, decimals=decimals))

            print("\nDelta (after - before):")
            display(self.values_table(self.last_delta_grid, decimals=decimals))

        self.render(
            title=f"After sweep {self.iteration}",
            show_policy=show_policy,
            figsize=figsize,
            highlight_changed=True,
        )

    def run_n_steps(self, n=1, verbose=True):
        for _ in range(n):
            if self.converged:
                break
            self.iterate_once(verbose=verbose)
        return self.values.copy()

    def run_to_convergence(self, max_iterations=None, verbose=True):
        if max_iterations is None:
            max_iterations = self.max_iterations

        while self.iteration < max_iterations and not self.converged:
            self.iterate_once(verbose=verbose)

        if self.converged:
            print(f"Stopped after {self.iteration} sweeps because the value function converged.")
        else:
            print(f"Stopped at the safety cap of {max_iterations} sweeps.")

        return self.values.copy()

    def render(self, title="Current Value Function", show_policy=False, figsize=(14, 6), highlight_changed=False):
        fig, (ax_grid, ax_err) = plt.subplots(1, 2, figsize=figsize)

        ax_grid.imshow(self.color_grid, origin="upper")
        ax_grid.set_xticks(range(self.grid_size), [f"c{j}" for j in range(self.grid_size)])
        ax_grid.set_yticks(range(self.grid_size), [f"r{i}" for i in range(self.grid_size)])
        ax_grid.set_xticks(np.arange(-0.5, self.grid_size, 1), minor=True)
        ax_grid.set_yticks(np.arange(-0.5, self.grid_size, 1), minor=True)
        ax_grid.grid(which="minor", color="white", linewidth=2)
        ax_grid.tick_params(which="minor", bottom=False, left=False)
        ax_grid.set_title(f"{title}  |  sweep = {self.iteration}", fontsize=18)

        for r in range(self.grid_size):
            for c in range(self.grid_size):
                reward = float(self.rewards[r, c])

                ax_grid.text(
                    c, r - 0.18, self._reward_label(reward),
                    ha="center", va="center",
                    color=("yellow" if reward == 1.0 else "white"),
                    fontsize=12, linespacing=1.0
                )

                value_color = "white"
                font_weight = "normal"
                if highlight_changed and self.iteration > 0 and abs(self.last_delta_grid[r, c]) > 1e-12:
                    value_color = "#8ef0ff"
                    font_weight = "bold"

                ax_grid.text(
                    c, r + 0.22, f"{self.values[r, c]:.3f}",
                    ha="center", va="center", color=value_color, fontsize=18, weight=font_weight
                )

                if (r, c) == self.start_state:
                    ax_grid.text(
                        c - 0.42, r + 0.38, "Start",
                        ha="left", va="center", color="#5ee6ff",
                        fontsize=13, weight="bold"
                    )

                if show_policy and (r, c) != self.goal_state:
                    best_indices, _ = self.greedy_actions((r, c))
                    offsets = {
                        1: [(0, -0.02)],
                        2: [(-0.14, -0.02), (0.14, -0.02)],
                        3: [(-0.18, -0.02), (0, -0.02), (0.18, -0.02)],
                        4: [(-0.18, -0.12), (0.18, -0.12), (-0.18, 0.08), (0.18, 0.08)],
                    }
                    for idx, act in enumerate(best_indices):
                        dx, dy = offsets[len(best_indices)][idx]
                        ax_grid.text(
                            c + dx, r, ARROWS[act],
                            ha="center", va="center",
                            color="gold", fontsize=26, weight="bold"
                        )
                elif show_policy and (r, c) == self.goal_state:
                    ax_grid.scatter([c], [r], s=90, c="yellow", edgecolors="black")

                if highlight_changed and self.iteration > 0 and abs(self.last_delta_grid[r, c]) > 1e-12:
                    rect = plt.Rectangle((c - 0.48, r - 0.48), 0.96, 0.96, fill=False, linewidth=2.0, edgecolor="#8ef0ff")
                    ax_grid.add_patch(rect)

        ax_err.set_title("Max Error per Sweep")
        ax_err.set_xlabel("Iteration")
        ax_err.set_ylabel("Max Error")
        ax_err.set_xlim(0, max(self.max_iterations, max(1, self.iteration)))
        ymax = max(1.2, max(self.error_history, default=0) * 1.1)
        ax_err.set_ylim(0, ymax)
        ax_err.grid(alpha=0.3)
        ax_err.axhline(self.epsilon, linestyle="--", linewidth=2, color="red", label=f"threshold = {self.epsilon}")

        if self.error_history:
            xs = np.arange(1, len(self.error_history) + 1)
            ys = np.array(self.error_history, dtype=float)
            ax_err.plot(xs, ys, marker="o")
            ax_err.legend(loc="upper right")

        status_lines = [
            f"Phase: {self.current_phase}",
            f"Sweep: {self.iteration}",
            f"Converged: {self.converged}",
            f"Last max error: {'--' if self.last_max_error is None else f'{self.last_max_error:.4f}'}",
        ]
        ax_err.text(
            0.04, 0.04,
            "\n".join(status_lines),
            transform=ax_err.transAxes,
            fontsize=13,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.75),
        )

        plt.tight_layout()
        plt.show()

    def plot_agent_run(self, max_steps=20):
        if not self.converged:
            print("Warning: values have not converged yet, so this is an agent run on the current value function, not the final one.")

        path = [self.start_state]
        visited = {self.start_state}
        current = self.start_state
        stop_reason = "max_steps"

        tie_order = {1: 0, 3: 1, 0: 2, 2: 3}  # Down, Right, Up, Left

        for _ in range(max_steps):
            if current == self.goal_state:
                stop_reason = "goal"
                break

            r, c = current
            best_indices, _ = self.greedy_actions(current)

            if not best_indices:
                stop_reason = "terminal"
                break

            candidates = []
            for action_idx in best_indices:
                tr = self._transition(r, c, action_idx)
                nr, nc = tr["next_state"]
                manhattan = abs(nr - self.goal_state[0]) + abs(nc - self.goal_state[1])

                candidates.append((
                    tr["q"],
                    tr["next_value"],
                    -manhattan,
                    -int((nr, nc) in visited),
                    -tie_order[action_idx],
                    action_idx,
                    (nr, nc),
                ))

            candidates.sort(reverse=True)
            next_state = candidates[0][6]

            path.append(next_state)

            if next_state == current or next_state in visited:
                current = next_state
                stop_reason = "loop"
                break

            current = next_state
            visited.add(current)

        if current == self.goal_state:
            stop_reason = "goal"

        title = "Agent Run on Final Value Function" if self.converged else "Agent Run on Current Value Function"

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(self.color_grid, origin="upper")
        ax.set_xticks(range(self.grid_size), [f"c{j}" for j in range(self.grid_size)])
        ax.set_yticks(range(self.grid_size), [f"r{i}" for i in range(self.grid_size)])
        ax.set_xticks(np.arange(-0.5, self.grid_size, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.grid_size, 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=2)
        ax.tick_params(which="minor", bottom=False, left=False)
        ax.set_title(title, fontsize=18)

        for r in range(self.grid_size):
            for c in range(self.grid_size):
                reward = float(self.rewards[r, c])
                ax.text(
                    c, r - 0.18, self._reward_label(reward),
                    ha="center", va="center",
                    color=("yellow" if reward == 1.0 else "white"),
                    fontsize=12, linespacing=1.0
                )
                ax.text(
                    c, r + 0.22, f"{self.values[r, c]:.3f}",
                    ha="center", va="center", color="white", fontsize=18
                )

        xs = [c for _, c in path]
        ys = [r for r, _ in path]

        ax.plot(xs, ys, marker="o", linewidth=3, alpha=0.9, label="Path")
        ax.scatter([self.start_state[1]], [self.start_state[0]], s=220, label="Start")
        end_x, end_y = path[-1][1], path[-1][0]
        if path[-1] == self.start_state:
            end_x += 0.18
            end_y += -0.18
        ax.scatter([end_x], [end_y], s=220, label="End")

        ax.legend(loc="upper left")
        plt.tight_layout()
        plt.show()

        print(f"Path: {path}")
        print(f"Stop reason: {stop_reason}")
        print(f"Number of moves used: {max(0, len(path) - 1)}")


## 2. Create the demo object

Run this cell once at the start.

A good habit is to keep using the **same** `demo` object while learning.  
If you create a brand-new `demo` again, the values go back to all zeros.


In [ ]:
demo = ValueIterationExplorer(gamma=0.9, epsilon=0.1, seed=42, max_iterations=40)
demo.show_status()
demo.render(title="Initial Value Grid")


## 3. Recreate the video's one-step lookahead example

This inspects state `(3, 4)`, the cell right above the goal.

Look at the `q_value` column and ask:

> Which action is best, and why?


In [ ]:
demo.inspect_state((3, 4))


## 4. Do one Bellman sweep and immediately display the result

Use this command when you want to avoid confusion.

`step_and_render()` does three things in one place:

1. updates the values  
2. prints **before / after / delta** tables  
3. redraws the grid and error chart

That makes it much easier to verify that the values are really changing.


In [ ]:
demo.step_and_render()


## 5. Run one more sweep

Watch how the larger positive values near the goal begin to spread farther across the grid.


In [ ]:
demo.step_and_render()


## 6. Inspect the current value table directly

Sometimes the table is easier to read than the plot.


In [ ]:
display(demo.values_table())


## 7. Inspect a state far from the goal

Try `(0, 0)` and compare it with `(3, 4)`.

This helps you see why information must travel backward through many sweeps.


In [ ]:
demo.inspect_state((0, 0))


## 8. Run a few more sweeps

This cell runs **3 more sweeps** and then shows the result.

You should see the values continue to change unless convergence has already been reached.


In [ ]:
demo.run_n_steps(3, verbose=True)
display(demo.values_table())
demo.render(title="After 5 Total Sweeps", highlight_changed=True)


## 9. Continue until convergence

The stopping rule is:

`max error < epsilon`

Here `epsilon = 0.1`.


In [ ]:
demo.run_to_convergence(max_iterations=40, verbose=False)
demo.show_status()
demo.render(title="Converged Value Function", highlight_changed=True)


## 10. Extract the greedy policy from the current value function

These arrows show the best action(s) under the current value table.


In [ ]:
demo.render(title="Greedy Policy from the Current Values", show_policy=True)
display(demo.policy_table())


## 11. Let the agent follow the current value function

If convergence is complete, this is effectively the final run.

If convergence is **not** complete, the notebook will warn you and label the plot as **Current Value Function** instead of **Final Value Function**.


In [ ]:
demo.plot_agent_run(max_steps=20)


## 12. Reset the environment

Run this whenever you want to start over from the all-zero value table.


In [ ]:
demo.reset(seed=42)
demo.show_status()
demo.render(title="Reset State")


## 13. Suggested student exercises

1. Change the inspected state from `(3, 4)` to `(2, 4)`, `(2, 3)`, and `(0, 0)`.  
2. Run one sweep at a time and describe which region of the grid changes first.  
3. Compare the value table after sweep 1, sweep 2, and sweep 5.  
4. Change `epsilon` to `0.01` and see how many more sweeps are needed.  
5. Explain why the goal state's own value is fixed at `0.0`, even though entering the goal gives reward `+1.0`.
